In [1]:

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


In [2]:
students_marks_path = Path(r"C:\Users\User\Desktop\Final Year Project\Data\Students_Marks.xlsx")

xls = pd.ExcelFile(students_marks_path)
sheet_names = xls.sheet_names
sheet_names


['CP1_Y1S1',
 'CP2_Y1S2',
 'CP3_Y2S1',
 'CP4_Y2S2',
 'CP5_Y3S1',
 'CP6_Y3S2',
 'CP7_Y4S1',
 'CP8_Y4S2']

In [ ]:
# load checkpoint 
def load_checkpoint(sheet_name: str) -> pd.DataFrame:
    df = pd.read_excel(students_marks_path, sheet_name=sheet_name)
    
    # Keep rows with label
    df = df.dropna(subset=["RISK_BAND"]).copy()
    
    # Standardize target text JIC
    df["RISK_BAND"] = df["RISK_BAND"].astype(str).str.strip()
    
    return df

df_cp1 = load_checkpoint("CP1_Y1S1")
df_cp1.shape, df_cp1["RISK_BAND"].value_counts()


((556, 31),
 RISK_BAND
 Moderate     193
 Risk         152
 Safe         151
 Very Safe     44
 High Risk     16
 Name: count, dtype: int64)

In [ ]:
# features (X) and label (y)
DROP_COLS = ["REGNO", "RISK_BAND", "checkpoint_sheet"]  # checkpoint_sheet is text

def split_xy(df: pd.DataFrame):
    y = df["RISK_BAND"].astype(str).str.strip()

    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # Keep only numeric columns 
    X = X.select_dtypes(include=[np.number])

    return X, y

In [ ]:

# Baselines :
models = {
    "LogReg": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", multi_class="auto"))
    ]),
    
    "RandomForest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            class_weight="balanced_subsample"
        ))
    ]),
    
    "HistGB": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            random_state=42
        ))
    ]),
}


In [ ]:

# Use macro F1 for multiclass imbalance 
F1_AVG = "macro"   

def evaluate_one_checkpoint(df: pd.DataFrame, test_size=0.2, random_state=42):
    X, y = split_xy(df)

    # class proportions similar in train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    results = []
    fitted = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average=F1_AVG)

        results.append({
            "model": name,
            "accuracy": acc,
            f"f1_{F1_AVG}": f1,
            "n_train": len(X_train),
            "n_test": len(X_test)
        })
        fitted[name] = (model, X_test, y_test, y_pred)

    results_df = pd.DataFrame(results).sort_values(by=[f"f1_{F1_AVG}", "accuracy"], ascending=False)
    return results_df, fitted


In [ ]:
# CP1_Y1S1
df_cp1 = load_checkpoint("CP1_Y1S1")
res_cp1, fitted_cp1 = evaluate_one_checkpoint(df_cp1)

res_cp1


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='

,model,accuracy,f1_macro,n_train,n_test
1,RandomForest,0.500000,0.396137,444,112
2,HistGB,0.500000,0.386737,444,112
0,LogReg,0.401786,0.342262,444,112


In [ ]:
# report for the best CP1 model
best_model_name = res_cp1.iloc[0]["model"]
model, X_test, y_test, y_pred = fitted_cp1[best_model_name]

print("Best CP1 model:", best_model_name)
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Best CP1 model: RandomForest
              precision    recall  f1-score   support

   High Risk       0.00      0.00      0.00         3
    Moderate       0.43      0.41      0.42        39
        Risk       0.58      0.61      0.59        31
        Safe       0.49      0.60      0.54        30
   Very Safe       0.60      0.33      0.43         9

    accuracy                           0.50       112
   macro avg       0.42      0.39      0.40       112
weighted avg       0.49      0.50      0.49       112

Confusion matrix:
 [[ 0  2  1  0  0]
 [ 0 16 11 12  0]
 [ 0 10 19  2  0]
 [ 0  8  2 18  2]
 [ 0  1  0  5  3]]


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.

In [13]:
#  (CP1..CP8)
cp_sheets = [s for s in sheet_names if s.startswith("CP")]
cp_sheets


['CP1_Y1S1',
 'CP2_Y1S2',
 'CP3_Y2S1',
 'CP4_Y2S2',
 'CP5_Y3S1',
 'CP6_Y3S2',
 'CP7_Y4S1',
 'CP8_Y4S2']

In [ ]:
# Evaluate rank by F1 then Accuracy
all_results = []

for sh in cp_sheets:
    df_cp = load_checkpoint(sh)
    res_df, _ = evaluate_one_checkpoint(df_cp)

    #best model for that checkpoint
    best_row = res_df.iloc[0].to_dict()
    best_row["checkpoint"] = sh
    all_results.append(best_row)

results_all = pd.DataFrame(all_results)

results_all["cp_num"] = results_all["checkpoint"].str.extract(r"CP(\d+)").astype(int)

results_all_sorted = results_all.sort_values(
    by=[f"f1_{F1_AVG}", "accuracy", "cp_num"],
    ascending=[False, False, True]
)

results_all_sorted


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:577: UserWarning: Skipping features without any observed values: ['wavg_mark_L4']. At least one non-missing value is needed for imputation with strategy='

,model,accuracy,f1_macro,n_train,n_test,checkpoint,cp_num
7,LogReg,0.741935,0.748838,616,155,CP8_Y4S2,8
6,LogReg,0.690323,0.706387,616,155,CP7_Y4S1,7
5,LogReg,0.625806,0.632363,616,155,CP6_Y3S2,6
4,LogReg,0.632258,0.610848,616,155,CP5_Y3S1,5
3,LogReg,0.600000,0.579631,616,155,CP4_Y2S2,4
2,LogReg,0.600000,0.551185,616,155,CP3_Y2S1,3
1,HistGB,0.535484,0.502799,616,155,CP2_Y1S2,2
0,RandomForest,0.500000,0.396137,444,112,CP1_Y1S1,1


In [ ]:
#  Earliest checkpoint among the best performers
best_f1 = results_all_sorted.iloc[0][f"f1_{F1_AVG}"]

# Keep all checkpoints within a tiny tolerance of the best 
TOL = 0.005  
top_group = results_all_sorted[results_all_sorted[f"f1_{F1_AVG}"] >= best_f1 - TOL].copy()

# choose earliest checkpoint 
top_group = top_group.sort_values(by=["cp_num", f"f1_{F1_AVG}", "accuracy"], ascending=[True, False, False])

top_group.head(10), best_f1


(    model  accuracy  f1_macro  n_train  n_test checkpoint  cp_num
 7  LogReg  0.741935  0.748838      616     155   CP8_Y4S2       8,
 0.7488375025614351)

In [ ]:

best_overall = results_all_sorted.iloc[0]
best_early   = top_group.iloc[0]

print("=== Best overall checkpoint (max F1) ===")
print(best_overall[["checkpoint", "model", f"f1_{F1_AVG}", "accuracy"]])

print("\n=== Earliest checkpoint within tolerance of best F1 ===")
print(best_early[["checkpoint", "model", f"f1_{F1_AVG}", "accuracy"]])


=== Best overall checkpoint (max F1) ===
checkpoint    CP8_Y4S2
model           LogReg
f1_macro      0.748838
accuracy      0.741935
Name: 7, dtype: object

=== Earliest checkpoint within tolerance of best F1 ===
checkpoint    CP8_Y4S2
model           LogReg
f1_macro      0.748838
accuracy      0.741935
Name: 7, dtype: object


pycaret

In [ ]:

import pandas as pd
import numpy as np

from pycaret.classification import ClassificationExperiment
from sklearn.metrics import f1_score

file_path = r"C:\Users\User\Desktop\Final Year Project\Data\Students_Marks.xlsx"


In [ ]:
# load checkpoint sheet
def load_cp(sheet_name: str) -> pd.DataFrame:
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df = df.dropna(subset=["RISK_BAND"]).copy()

    return df


In [ ]:
# Compare models CP1 only ranked by Macro F1
df_cp1 = pd.read_excel(file_path, sheet_name="CP1_Y1S1")
df_cp1 = df_cp1.dropna(subset=["RISK_BAND"]).copy()

exp = ClassificationExperiment()
exp.setup(
    data=df_cp1,
    target="RISK_BAND",
    ignore_features=["REGNO", "checkpoint_sheet"],
    session_id=42,
    fold=5,
    fold_shuffle=True,
    verbose=False
)

best_cp1 = exp.compare_models(sort="F1")   # <-- ranked by F1
lb_cp1 = exp.pull()

lb_cp1.head(15)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.5165,0.0000,0.5165,0.5234,0.5126,0.3202,0.3222,0.1920
rf,Random Forest Classifier,0.5193,0.7609,0.5193,0.5213,0.5103,0.3201,0.3244,0.1140
lda,Linear Discriminant Analysis,0.4988,0.0000,0.4988,0.5224,0.4977,0.2990,0.3016,0.0280
lightgbm,Light Gradient Boosting Machine,0.5014,0.7291,0.5014,0.4796,0.4864,0.2975,0.2997,0.7900
ridge,Ridge Classifier,0.4910,0.0000,0.4910,0.4924,0.4713,0.2734,0.2778,0.0220
et,Extra Trees Classifier,0.4783,0.7350,0.4783,0.4812,0.4701,0.2690,0.2714,0.1600
knn,K Neighbors Classifier,0.4626,0.6822,0.4626,0.4791,0.4451,0.2239,0.2328,0.0340
gbc,Gradient Boosting Classifier,0.4449,0.0000,0.4449,0.4309,0.4345,0.2233,0.2247,0.4040
dt,Decision Tree Classifier,0.4242,0.5975,0.4242,0.4142,0.4170,0.2000,0.2010,0.0220
ada,Ada Boost Classifier,0.4139,0.0000,0.4139,0.4114,0.3736,0.2075,0.2220,0.1300


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.5165,0.0000,0.5165,0.5234,0.5126,0.3202,0.3222,0.192
rf,Random Forest Classifier,0.5193,0.7609,0.5193,0.5213,0.5103,0.3201,0.3244,0.114
lda,Linear Discriminant Analysis,0.4988,0.0000,0.4988,0.5224,0.4977,0.2990,0.3016,0.028
lightgbm,Light Gradient Boosting Machine,0.5014,0.7291,0.5014,0.4796,0.4864,0.2975,0.2997,0.790
ridge,Ridge Classifier,0.4910,0.0000,0.4910,0.4924,0.4713,0.2734,0.2778,0.022
et,Extra Trees Classifier,0.4783,0.7350,0.4783,0.4812,0.4701,0.2690,0.2714,0.160
knn,K Neighbors Classifier,0.4626,0.6822,0.4626,0.4791,0.4451,0.2239,0.2328,0.034
gbc,Gradient Boosting Classifier,0.4449,0.0000,0.4449,0.4309,0.4345,0.2233,0.2247,0.404
dt,Decision Tree Classifier,0.4242,0.5975,0.4242,0.4142,0.4170,0.2000,0.2010,0.022
ada,Ada Boost Classifier,0.4139,0.0000,0.4139,0.4114,0.3736,0.2075,0.2220,0.130


In [ ]:
#  CP1 leaderboard sorted by f1_macro
leaderboard_cp1.sort_values("F1 Macro", ascending=False).head(30)


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
lr,Logistic Regression,0.5165,0.0000,0.5165,0.5234,0.5126,0.3202,0.3222,0.0,2.068
knn,K Neighbors Classifier,0.4626,0.6822,0.4626,0.4791,0.4451,0.2239,0.2328,0.0,0.608
nb,Naive Bayes,0.3083,0.6618,0.3083,0.3976,0.2772,0.1469,0.1684,0.0,0.018
dt,Decision Tree Classifier,0.4242,0.5975,0.4242,0.4142,0.4170,0.2000,0.2010,0.0,0.016
svm,SVM - Linear Kernel,0.3033,0.0000,0.3033,0.0932,0.1422,0.0000,0.0000,0.0,0.036
ridge,Ridge Classifier,0.4910,0.0000,0.4910,0.4924,0.4713,0.2734,0.2778,0.0,0.020
rf,Random Forest Classifier,0.5193,0.7609,0.5193,0.5213,0.5103,0.3201,0.3244,0.0,0.084
qda,Quadratic Discriminant Analysis,0.3522,0.0000,0.3522,0.1751,0.2047,0.0163,0.0210,0.0,0.020
ada,Ada Boost Classifier,0.4139,0.0000,0.4139,0.4114,0.3736,0.2075,0.2220,0.0,0.046
gbc,Gradient Boosting Classifier,0.4449,0.0000,0.4449,0.4309,0.4345,0.2233,0.2247,0.0,0.296


In [4]:
xls = pd.ExcelFile(file_path)
cp_sheets = [s for s in xls.sheet_names if s.startswith("CP")]
cp_sheets

['CP1_Y1S1',
 'CP2_Y1S2',
 'CP3_Y2S1',
 'CP4_Y2S2',
 'CP5_Y3S1',
 'CP6_Y3S2',
 'CP7_Y4S1',
 'CP8_Y4S2']

In [ ]:
# record the best model best macro F1
results = []

for sh in cp_sheets:
    df = pd.read_excel(file_path, sheet_name=sh).dropna(subset=["RISK_BAND"]).copy()

    exp = ClassificationExperiment()
    exp.setup(
        data=df,
        target="RISK_BAND",
        ignore_features=["REGNO", "checkpoint_sheet"],
        session_id=42,
        fold=5,
        fold_shuffle=True,
        verbose=False
    )

    best = exp.compare_models(sort="F1")
    lb = exp.pull()


    lb_sorted = lb.sort_values(by=["F1", "Accuracy"], ascending=False)
    top = lb_sorted.iloc[0]

    cp_num = int(sh.split("_")[0].replace("CP", ""))

    results.append({
        "checkpoint": sh,
        "cp_num": cp_num,
        "best_model": top["Model"],
        "f1": float(top["F1"]),
        "accuracy": float(top["Accuracy"])
    })

results_df = pd.DataFrame(results).sort_values(
    by=["f1", "accuracy", "cp_num"],
    ascending=[False, False, True]
).reset_index(drop=True)

results_df


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.5165,0.0000,0.5165,0.5234,0.5126,0.3202,0.3222,1.8340
rf,Random Forest Classifier,0.5193,0.7609,0.5193,0.5213,0.5103,0.3201,0.3244,0.0920
lda,Linear Discriminant Analysis,0.4988,0.0000,0.4988,0.5224,0.4977,0.2990,0.3016,0.0220
lightgbm,Light Gradient Boosting Machine,0.5014,0.7291,0.5014,0.4796,0.4864,0.2975,0.2997,0.4300
ridge,Ridge Classifier,0.4910,0.0000,0.4910,0.4924,0.4713,0.2734,0.2778,0.0180
et,Extra Trees Classifier,0.4783,0.7350,0.4783,0.4812,0.4701,0.2690,0.2714,0.0960
knn,K Neighbors Classifier,0.4626,0.6822,0.4626,0.4791,0.4451,0.2239,0.2328,0.8580
gbc,Gradient Boosting Classifier,0.4449,0.0000,0.4449,0.4309,0.4345,0.2233,0.2247,0.3960
dt,Decision Tree Classifier,0.4242,0.5975,0.4242,0.4142,0.4170,0.2000,0.2010,0.0240
ada,Ada Boost Classifier,0.4139,0.0000,0.4139,0.4114,0.3736,0.2075,0.2220,0.0600


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.5584,0.7925,0.5584,0.5468,0.5490,0.3668,0.3692,0.1020
lda,Linear Discriminant Analysis,0.5456,0.0000,0.5456,0.5563,0.5438,0.3503,0.3541,0.0240
et,Extra Trees Classifier,0.5510,0.7935,0.5510,0.5396,0.5414,0.3551,0.3579,0.1140
lr,Logistic Regression,0.5399,0.0000,0.5399,0.5454,0.5380,0.3506,0.3531,0.1040
knn,K Neighbors Classifier,0.5380,0.7589,0.5380,0.5377,0.5272,0.3292,0.3350,0.0180
gbc,Gradient Boosting Classifier,0.5214,0.0000,0.5214,0.5171,0.5142,0.3179,0.3188,0.6040
lightgbm,Light Gradient Boosting Machine,0.5102,0.7618,0.5102,0.4990,0.5013,0.2974,0.2989,0.6180
ridge,Ridge Classifier,0.5196,0.0000,0.5196,0.4770,0.4945,0.3006,0.3040,0.0160
dt,Decision Tree Classifier,0.4360,0.6035,0.4360,0.4452,0.4361,0.2083,0.2097,0.0180
ada,Ada Boost Classifier,0.4506,0.0000,0.4506,0.4495,0.4252,0.2294,0.2406,0.0580


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.6270,0.8448,0.6270,0.6195,0.6185,0.4640,0.4667,0.1020
gbc,Gradient Boosting Classifier,0.6104,0.0000,0.6104,0.6118,0.6066,0.4444,0.4462,0.5380
et,Extra Trees Classifier,0.6140,0.8449,0.6140,0.6032,0.6043,0.4459,0.4488,0.0860
lightgbm,Light Gradient Boosting Machine,0.6085,0.8237,0.6085,0.6024,0.6033,0.4431,0.4443,0.6220
lda,Linear Discriminant Analysis,0.6048,0.0000,0.6048,0.6148,0.6031,0.4395,0.4422,0.0160
lr,Logistic Regression,0.5863,0.0000,0.5863,0.5801,0.5813,0.4158,0.4170,0.0840
knn,K Neighbors Classifier,0.5714,0.7930,0.5714,0.5817,0.5598,0.3751,0.3824,0.0200
nb,Naive Bayes,0.5399,0.7988,0.5399,0.5990,0.5313,0.3723,0.3852,0.0180
dt,Decision Tree Classifier,0.5326,0.6709,0.5326,0.5407,0.5312,0.3449,0.3475,0.0200
ridge,Ridge Classifier,0.5232,0.0000,0.5232,0.4820,0.4972,0.3043,0.3077,0.0140


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.6846,0.0000,0.6846,0.6932,0.6820,0.5520,0.5555,0.0220
et,Extra Trees Classifier,0.6734,0.8871,0.6734,0.6586,0.6627,0.5318,0.5336,0.0900
lightgbm,Light Gradient Boosting Machine,0.6715,0.8733,0.6715,0.6626,0.6627,0.5316,0.5339,0.5100
gbc,Gradient Boosting Classifier,0.6604,0.0000,0.6604,0.6543,0.6552,0.5192,0.5205,0.5460
rf,Random Forest Classifier,0.6622,0.8867,0.6622,0.6555,0.6544,0.5174,0.5191,0.0980
ada,Ada Boost Classifier,0.6232,0.0000,0.6232,0.5948,0.5841,0.4493,0.4690,0.0620
lr,Logistic Regression,0.5807,0.0000,0.5807,0.5759,0.5759,0.4050,0.4066,0.0960
dt,Decision Tree Classifier,0.5695,0.6972,0.5695,0.5700,0.5679,0.3941,0.3949,0.0160
knn,K Neighbors Classifier,0.5807,0.8163,0.5807,0.5846,0.5643,0.3879,0.3959,0.0200
ridge,Ridge Classifier,0.5621,0.0000,0.5621,0.5105,0.5325,0.3652,0.3691,0.0140


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.7273,0.0000,0.7273,0.7400,0.7279,0.6147,0.6175,0.0160
gbc,Gradient Boosting Classifier,0.7217,0.0000,0.7217,0.7199,0.7168,0.6048,0.6066,0.5740
rf,Random Forest Classifier,0.7161,0.9027,0.7161,0.7093,0.7076,0.5938,0.5961,0.1260
et,Extra Trees Classifier,0.7106,0.8969,0.7106,0.6990,0.7000,0.5842,0.5862,0.0940
lightgbm,Light Gradient Boosting Machine,0.6956,0.8917,0.6956,0.6874,0.6890,0.5670,0.5682,0.5500
lr,Logistic Regression,0.6642,0.0000,0.6642,0.6614,0.6595,0.5235,0.5247,0.0860
dt,Decision Tree Classifier,0.5919,0.7136,0.5919,0.5954,0.5913,0.4249,0.4260,0.0200
knn,K Neighbors Classifier,0.6011,0.8252,0.6011,0.5925,0.5856,0.4217,0.4282,0.0200
ridge,Ridge Classifier,0.5845,0.0000,0.5845,0.5415,0.5589,0.3988,0.4023,0.0200
qda,Quadratic Discriminant Analysis,0.5863,0.0000,0.5863,0.5378,0.5526,0.3972,0.4055,0.0180


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.7253,0.0000,0.7253,0.7324,0.7243,0.6125,0.6149,0.0180
rf,Random Forest Classifier,0.7272,0.9030,0.7272,0.7123,0.7140,0.6069,0.6093,0.1100
et,Extra Trees Classifier,0.6994,0.8987,0.6994,0.6861,0.6872,0.5674,0.5697,0.0880
gbc,Gradient Boosting Classifier,0.6901,0.0000,0.6901,0.6874,0.6837,0.5570,0.5595,0.6420
lightgbm,Light Gradient Boosting Machine,0.6882,0.8816,0.6882,0.6888,0.6813,0.5559,0.5584,0.6040
lr,Logistic Regression,0.6493,0.0000,0.6493,0.6535,0.6476,0.5041,0.5055,0.1000
dt,Decision Tree Classifier,0.6159,0.7290,0.6159,0.6122,0.6127,0.4564,0.4571,0.0300
knn,K Neighbors Classifier,0.6066,0.8137,0.6066,0.6077,0.5889,0.4246,0.4337,0.0180
ridge,Ridge Classifier,0.5955,0.0000,0.5955,0.5488,0.5685,0.4154,0.4192,0.0160
nb,Naive Bayes,0.5344,0.8071,0.5344,0.5947,0.5142,0.3813,0.4129,0.0140


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.7587,0.0000,0.7587,0.7628,0.7572,0.6596,0.6610,0.0180
lightgbm,Light Gradient Boosting Machine,0.7328,0.9144,0.7328,0.7315,0.7271,0.6208,0.6230,0.6340
et,Extra Trees Classifier,0.7365,0.9193,0.7365,0.7313,0.7262,0.6214,0.6235,0.0860
rf,Random Forest Classifier,0.7365,0.9192,0.7365,0.7373,0.7249,0.6208,0.6247,0.1360
gbc,Gradient Boosting Classifier,0.7253,0.0000,0.7253,0.7249,0.7198,0.6104,0.6124,0.7460
lr,Logistic Regression,0.7069,0.0000,0.7069,0.7104,0.7016,0.5873,0.5903,0.1380
knn,K Neighbors Classifier,0.6308,0.8311,0.6308,0.6523,0.6204,0.4632,0.4710,0.0260
dt,Decision Tree Classifier,0.6214,0.7317,0.6214,0.6234,0.6183,0.4659,0.4677,0.0220
ridge,Ridge Classifier,0.6382,0.0000,0.6382,0.5980,0.6101,0.4809,0.4865,0.0240
nb,Naive Bayes,0.5603,0.8267,0.5603,0.6292,0.5382,0.4082,0.4352,0.0240


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8145,0.9510,0.8145,0.8166,0.8067,0.7344,0.7368,0.1320
lightgbm,Light Gradient Boosting Machine,0.8070,0.9465,0.8070,0.8151,0.8062,0.7267,0.7281,0.6040
et,Extra Trees Classifier,0.8108,0.9518,0.8108,0.8145,0.8037,0.7297,0.7325,0.0920
lda,Linear Discriminant Analysis,0.7940,0.0000,0.7940,0.8054,0.7926,0.7079,0.7115,0.0160
gbc,Gradient Boosting Classifier,0.7866,0.0000,0.7866,0.7949,0.7838,0.6981,0.7005,0.6000
lr,Logistic Regression,0.7458,0.0000,0.7458,0.7541,0.7451,0.6415,0.6436,0.0880
dt,Decision Tree Classifier,0.6771,0.7714,0.6771,0.6890,0.6758,0.5451,0.5488,0.0240
knn,K Neighbors Classifier,0.6605,0.8585,0.6605,0.6750,0.6446,0.5048,0.5125,0.0180
ridge,Ridge Classifier,0.6401,0.0000,0.6401,0.6056,0.6165,0.4835,0.4875,0.0220
nb,Naive Bayes,0.5975,0.8632,0.5975,0.6531,0.5798,0.4592,0.4834,0.0140


,checkpoint,cp_num,best_model,f1,accuracy
0,CP8_Y4S2,8,Random Forest Classifier,0.8067,0.8145
1,CP7_Y4S1,7,Linear Discriminant Analysis,0.7572,0.7587
2,CP5_Y3S1,5,Linear Discriminant Analysis,0.7279,0.7273
3,CP6_Y3S2,6,Linear Discriminant Analysis,0.7243,0.7253
4,CP4_Y2S2,4,Linear Discriminant Analysis,0.6820,0.6846
5,CP3_Y2S1,3,Random Forest Classifier,0.6185,0.6270
6,CP2_Y1S2,2,Random Forest Classifier,0.5490,0.5584
7,CP1_Y1S1,1,Logistic Regression,0.5126,0.5165


In [ ]:
#Pick earliest checkpoint near-best
best_f1 = results_df.loc[0, "f1_macro"]

TOL = 0.01  
near_best = results_df[results_df["f1_macro"] >= best_f1 - TOL].copy()
near_best = near_best.sort_values(by=["cp_num", "f1_macro", "accuracy"], ascending=[True, False, False])

print("Best overall:")
print(results_df.iloc[0])

print("\nEarliest checkpoint within tolerance:")
print(near_best.iloc[0])


Best overall:
checkpoint               CP8_Y4S2
cp_num                          8
best_model    Logistic Regression
f1_macro                      0.0
accuracy                   0.7458
Name: 0, dtype: object

Earliest checkpoint within tolerance:
checkpoint               CP1_Y1S1
cp_num                          1
best_model    Logistic Regression
f1_macro                      0.0
accuracy                   0.5165
Name: 7, dtype: object


In [ ]:
# Loop checkpoints and record best model per checkpoint
from pycaret.classification import ClassificationExperiment
from sklearn.metrics import f1_score
import pandas as pd

results = []

for sh in cp_sheets:
    df = load_cp(sh)

    exp = ClassificationExperiment()
    exp.setup(
        data=df,
        target="RISK_BAND",
        ignore_features=["REGNO", "checkpoint_sheet"],
        session_id=42,
        fold=5,
        fold_shuffle=True,
        verbose=False
    )

   
    exp.add_metric(
        id="f1_macro",
        name="F1 Macro",
        score_func=f1_score,
        average="macro"
    )

   
    best = exp.compare_models(sort="Accuracy")
    lb = exp.pull()

    
    lb_sorted = lb.sort_values(by=["Accuracy", "F1 Macro"], ascending=False)
    top = lb_sorted.iloc[0]

    cp_num = int(sh.split("_")[0].replace("CP", ""))

    results.append({
        "checkpoint": sh,
        "cp_num": cp_num,
        "best_model": top["Model"],
        "accuracy": float(top["Accuracy"]),
        "f1_macro": float(top["F1 Macro"]),
    })

results_df = pd.DataFrame(results).sort_values(
    by=["accuracy", "f1_macro", "cp_num"],
    ascending=[False, False, True]
).reset_index(drop=True)

results_df


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
rf,Random Forest Classifier,0.5193,0.7609,0.5193,0.5213,0.5103,0.3201,0.3244,0.0000,0.1240
lr,Logistic Regression,0.5165,0.0000,0.5165,0.5234,0.5126,0.3202,0.3222,0.0000,1.3180
lightgbm,Light Gradient Boosting Machine,0.5014,0.7291,0.5014,0.4796,0.4864,0.2975,0.2997,0.0000,0.4260
lda,Linear Discriminant Analysis,0.4988,0.0000,0.4988,0.5224,0.4977,0.2990,0.3016,0.0000,0.0180
ridge,Ridge Classifier,0.4910,0.0000,0.4910,0.4924,0.4713,0.2734,0.2778,0.0000,0.0180
et,Extra Trees Classifier,0.4783,0.7350,0.4783,0.4812,0.4701,0.2690,0.2714,0.0000,0.0840
knn,K Neighbors Classifier,0.4626,0.6822,0.4626,0.4791,0.4451,0.2239,0.2328,0.0000,0.8060
gbc,Gradient Boosting Classifier,0.4449,0.0000,0.4449,0.4309,0.4345,0.2233,0.2247,0.0000,0.4140
dt,Decision Tree Classifier,0.4242,0.5975,0.4242,0.4142,0.4170,0.2000,0.2010,0.0000,0.0280
ada,Ada Boost Classifier,0.4139,0.0000,0.4139,0.4114,0.3736,0.2075,0.2220,0.0000,0.0640


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
rf,Random Forest Classifier,0.5584,0.7925,0.5584,0.5468,0.5490,0.3668,0.3692,0.0000,0.1060
et,Extra Trees Classifier,0.5510,0.7935,0.5510,0.5396,0.5414,0.3551,0.3579,0.0000,0.0860
lda,Linear Discriminant Analysis,0.5456,0.0000,0.5456,0.5563,0.5438,0.3503,0.3541,0.0000,0.0180
lr,Logistic Regression,0.5399,0.0000,0.5399,0.5454,0.5380,0.3506,0.3531,0.0000,0.1120
knn,K Neighbors Classifier,0.5380,0.7589,0.5380,0.5377,0.5272,0.3292,0.3350,0.0000,0.0200
gbc,Gradient Boosting Classifier,0.5214,0.0000,0.5214,0.5171,0.5142,0.3179,0.3188,0.0000,0.4840
ridge,Ridge Classifier,0.5196,0.0000,0.5196,0.4770,0.4945,0.3006,0.3040,0.0000,0.0160
lightgbm,Light Gradient Boosting Machine,0.5102,0.7618,0.5102,0.4990,0.5013,0.2974,0.2989,0.0000,0.6180
ada,Ada Boost Classifier,0.4506,0.0000,0.4506,0.4495,0.4252,0.2294,0.2406,0.0000,0.0580
dt,Decision Tree Classifier,0.4360,0.6035,0.4360,0.4452,0.4361,0.2083,0.2097,0.0000,0.0180


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
rf,Random Forest Classifier,0.6270,0.8448,0.6270,0.6195,0.6185,0.4640,0.4667,0.0000,0.1000
et,Extra Trees Classifier,0.6140,0.8449,0.6140,0.6032,0.6043,0.4459,0.4488,0.0000,0.0820
gbc,Gradient Boosting Classifier,0.6104,0.0000,0.6104,0.6118,0.6066,0.4444,0.4462,0.0000,0.5060
lightgbm,Light Gradient Boosting Machine,0.6085,0.8237,0.6085,0.6024,0.6033,0.4431,0.4443,0.0000,0.5760
lda,Linear Discriminant Analysis,0.6048,0.0000,0.6048,0.6148,0.6031,0.4395,0.4422,0.0000,0.0160
lr,Logistic Regression,0.5863,0.0000,0.5863,0.5801,0.5813,0.4158,0.4170,0.0000,0.1260
knn,K Neighbors Classifier,0.5714,0.7930,0.5714,0.5817,0.5598,0.3751,0.3824,0.0000,0.0200
nb,Naive Bayes,0.5399,0.7988,0.5399,0.5990,0.5313,0.3723,0.3852,0.0000,0.0180
dt,Decision Tree Classifier,0.5326,0.6709,0.5326,0.5407,0.5312,0.3449,0.3475,0.0000,0.0300
ridge,Ridge Classifier,0.5232,0.0000,0.5232,0.4820,0.4972,0.3043,0.3077,0.0000,0.0180


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
lda,Linear Discriminant Analysis,0.6846,0.0000,0.6846,0.6932,0.6820,0.5520,0.5555,0.0000,0.0200
et,Extra Trees Classifier,0.6734,0.8871,0.6734,0.6586,0.6627,0.5318,0.5336,0.0000,0.0960
lightgbm,Light Gradient Boosting Machine,0.6715,0.8733,0.6715,0.6626,0.6627,0.5316,0.5339,0.0000,0.5540
rf,Random Forest Classifier,0.6622,0.8867,0.6622,0.6555,0.6544,0.5174,0.5191,0.0000,0.1000
gbc,Gradient Boosting Classifier,0.6604,0.0000,0.6604,0.6543,0.6552,0.5192,0.5205,0.0000,0.5260
ada,Ada Boost Classifier,0.6232,0.0000,0.6232,0.5948,0.5841,0.4493,0.4690,0.0000,0.0620
lr,Logistic Regression,0.5807,0.0000,0.5807,0.5759,0.5759,0.4050,0.4066,0.0000,0.1100
knn,K Neighbors Classifier,0.5807,0.8163,0.5807,0.5846,0.5643,0.3879,0.3959,0.0000,0.0260
dt,Decision Tree Classifier,0.5695,0.6972,0.5695,0.5700,0.5679,0.3941,0.3949,0.0000,0.0200
ridge,Ridge Classifier,0.5621,0.0000,0.5621,0.5105,0.5325,0.3652,0.3691,0.0000,0.0160


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
lda,Linear Discriminant Analysis,0.7273,0.0000,0.7273,0.7400,0.7279,0.6147,0.6175,0.0000,0.0160
gbc,Gradient Boosting Classifier,0.7217,0.0000,0.7217,0.7199,0.7168,0.6048,0.6066,0.0000,0.5440
rf,Random Forest Classifier,0.7161,0.9027,0.7161,0.7093,0.7076,0.5938,0.5961,0.0000,0.0980
et,Extra Trees Classifier,0.7106,0.8969,0.7106,0.6990,0.7000,0.5842,0.5862,0.0000,0.1040
lightgbm,Light Gradient Boosting Machine,0.6956,0.8917,0.6956,0.6874,0.6890,0.5670,0.5682,0.0000,0.5160
lr,Logistic Regression,0.6642,0.0000,0.6642,0.6614,0.6595,0.5235,0.5247,0.0000,0.1060
knn,K Neighbors Classifier,0.6011,0.8252,0.6011,0.5925,0.5856,0.4217,0.4282,0.0000,0.0240
dt,Decision Tree Classifier,0.5919,0.7136,0.5919,0.5954,0.5913,0.4249,0.4260,0.0000,0.0200
qda,Quadratic Discriminant Analysis,0.5863,0.0000,0.5863,0.5378,0.5526,0.3972,0.4055,0.0000,0.0180
ridge,Ridge Classifier,0.5845,0.0000,0.5845,0.5415,0.5589,0.3988,0.4023,0.0000,0.0160


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
rf,Random Forest Classifier,0.7272,0.9030,0.7272,0.7123,0.7140,0.6069,0.6093,0.0000,0.1060
lda,Linear Discriminant Analysis,0.7253,0.0000,0.7253,0.7324,0.7243,0.6125,0.6149,0.0000,0.0160
et,Extra Trees Classifier,0.6994,0.8987,0.6994,0.6861,0.6872,0.5674,0.5697,0.0000,0.0880
gbc,Gradient Boosting Classifier,0.6901,0.0000,0.6901,0.6874,0.6837,0.5570,0.5595,0.0000,0.5960
lightgbm,Light Gradient Boosting Machine,0.6882,0.8816,0.6882,0.6888,0.6813,0.5559,0.5584,0.0000,0.5220
lr,Logistic Regression,0.6493,0.0000,0.6493,0.6535,0.6476,0.5041,0.5055,0.0000,0.1180
dt,Decision Tree Classifier,0.6159,0.7290,0.6159,0.6122,0.6127,0.4564,0.4571,0.0000,0.0200
knn,K Neighbors Classifier,0.6066,0.8137,0.6066,0.6077,0.5889,0.4246,0.4337,0.0000,0.0200
ridge,Ridge Classifier,0.5955,0.0000,0.5955,0.5488,0.5685,0.4154,0.4192,0.0000,0.0160
qda,Quadratic Discriminant Analysis,0.5475,0.0000,0.5475,0.5102,0.4958,0.3237,0.3470,0.0000,0.0320


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
lda,Linear Discriminant Analysis,0.7587,0.0000,0.7587,0.7628,0.7572,0.6596,0.6610,0.0000,0.0200
rf,Random Forest Classifier,0.7365,0.9192,0.7365,0.7373,0.7249,0.6208,0.6247,0.0000,0.1060
et,Extra Trees Classifier,0.7365,0.9193,0.7365,0.7313,0.7262,0.6214,0.6235,0.0000,0.0860
lightgbm,Light Gradient Boosting Machine,0.7328,0.9144,0.7328,0.7315,0.7271,0.6208,0.6230,0.0000,0.5980
gbc,Gradient Boosting Classifier,0.7253,0.0000,0.7253,0.7249,0.7198,0.6104,0.6124,0.0000,0.5940
lr,Logistic Regression,0.7069,0.0000,0.7069,0.7104,0.7016,0.5873,0.5903,0.0000,0.1180
ridge,Ridge Classifier,0.6382,0.0000,0.6382,0.5980,0.6101,0.4809,0.4865,0.0000,0.0160
knn,K Neighbors Classifier,0.6308,0.8311,0.6308,0.6523,0.6204,0.4632,0.4710,0.0000,0.0200
dt,Decision Tree Classifier,0.6214,0.7317,0.6214,0.6234,0.6183,0.4659,0.4677,0.0000,0.0240
nb,Naive Bayes,0.5603,0.8267,0.5603,0.6292,0.5382,0.4082,0.4352,0.0000,0.0220


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,F1 Macro,TT (Sec)
rf,Random Forest Classifier,0.8145,0.9510,0.8145,0.8166,0.8067,0.7344,0.7368,0.0000,0.1200
et,Extra Trees Classifier,0.8108,0.9518,0.8108,0.8145,0.8037,0.7297,0.7325,0.0000,0.1160
lightgbm,Light Gradient Boosting Machine,0.8070,0.9465,0.8070,0.8151,0.8062,0.7267,0.7281,0.0000,0.6200
lda,Linear Discriminant Analysis,0.7940,0.0000,0.7940,0.8054,0.7926,0.7079,0.7115,0.0000,0.0180
gbc,Gradient Boosting Classifier,0.7866,0.0000,0.7866,0.7949,0.7838,0.6981,0.7005,0.0000,0.6740
lr,Logistic Regression,0.7458,0.0000,0.7458,0.7541,0.7451,0.6415,0.6436,0.0000,0.1140
dt,Decision Tree Classifier,0.6771,0.7714,0.6771,0.6890,0.6758,0.5451,0.5488,0.0000,0.0240
knn,K Neighbors Classifier,0.6605,0.8585,0.6605,0.6750,0.6446,0.5048,0.5125,0.0000,0.0220
ridge,Ridge Classifier,0.6401,0.0000,0.6401,0.6056,0.6165,0.4835,0.4875,0.0000,0.0180
nb,Naive Bayes,0.5975,0.8632,0.5975,0.6531,0.5798,0.4592,0.4834,0.0000,0.0180


,checkpoint,cp_num,best_model,accuracy,f1_macro
0,CP8_Y4S2,8,Random Forest Classifier,0.8145,0.0
1,CP7_Y4S1,7,Linear Discriminant Analysis,0.7587,0.0
2,CP5_Y3S1,5,Linear Discriminant Analysis,0.7273,0.0
3,CP6_Y3S2,6,Random Forest Classifier,0.7272,0.0
4,CP4_Y2S2,4,Linear Discriminant Analysis,0.6846,0.0
5,CP3_Y2S1,3,Random Forest Classifier,0.6270,0.0
6,CP2_Y1S2,2,Random Forest Classifier,0.5584,0.0
7,CP1_Y1S1,1,Random Forest Classifier,0.5193,0.0


In [ ]:
#  Pick earliest checkpoint 
best_f1 = results_df.loc[0, "f1"]
TOL = 0.01  
near_best = results_df[results_df["f1"] >= best_f1 - TOL].copy()
near_best = near_best.sort_values(by=["cp_num", "f1", "accuracy"], ascending=[True, False, False])

print("Best overall by F1:")
print(results_df.iloc[0])

print("\nEarliest checkpoint within tolerance:")
print(near_best.iloc[0])

Best overall by F1:
checkpoint                    CP8_Y4S2
cp_num                               8
best_model    Random Forest Classifier
f1                              0.8067
accuracy                        0.8145
Name: 0, dtype: object

Earliest checkpoint within tolerance:
checkpoint                    CP8_Y4S2
cp_num                               8
best_model    Random Forest Classifier
f1                              0.8067
accuracy                        0.8145
Name: 0, dtype: object
